In [1]:
import pandas as pd
from sqlalchemy import create_engine
from urllib.parse import quote
from matplotlib import pyplot as plt
import seaborn as sns

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [2]:
# параметры подключения
username = "student"
password = "123456sql@"
host = "rc1b-cpsjckr2wfkuk61s.mdb.yandexcloud.net"
port = "6432"
database = "Northwind"

# Кодируем пароль
encoded_password = quote(password)
print(encoded_password)

# Формируем строку подключения
connection_string = f'postgresql+psycopg2://{username}:{encoded_password}@{host}:{port}/{database}'
engine = create_engine(connection_string)

123456sql%40


In [3]:
#sql запрос
query = """
with t1 as (
select seller_id, date(date_trunc('month', min(date_paid))) as cohort_month  --когорта определяется по дате выкупа заказа
from sandbox.orders o join  sandbox.order_details od on o.order_id = od.order_id 
group by seller_id
)
select 
cohort_month,
date(date_trunc('month',date_paid)) as  purchase_month,
row_number()over(partition by cohort_month order by date(date_trunc('month',date_paid))) purchase_month_number,
count(distinct od.seller_id) as sellers_paid,
sum(quantity * price) as gmv_paid,
sum(quantity * price * commission) as margin_paid, --добавлена маржа
sum(quantity * price)/count(distinct od.seller_id) as arpps_paid
from sandbox.orders o join  sandbox.order_details od on o.order_id = od.order_id 
join t1 on od.seller_id = t1.seller_id
group by cohort_month, date(date_trunc('month', date_paid))
order by cohort_month, date(date_trunc('month', date_paid))
"""
df = pd.read_sql_query(query, engine)
df.head()

,cohort_month,purchase_month,purchase_month_number,sellers_paid,gmv_paid,margin_paid,arpps_paid
0,2017-01-01,2017-01-01,1,176,82381.46,21244.7335,468.076477
1,2017-01-01,2017-02-01,2,141,108402.27,29019.1298,768.810426
2,2017-01-01,2017-03-01,3,142,174984.71,47042.2519,1232.286690
3,2017-01-01,2017-04-01,4,117,137618.62,35870.6588,1176.227521
4,2017-01-01,2017-05-01,5,119,189521.28,53845.3047,1592.615798


In [4]:
df_sellers_pivot = df.pivot_table(
    index='cohort_month',   # Строки - месяца когорты
    columns='purchase_month_number',  # Столбцы - месяцы покупок
    values='sellers_paid',  # Значения - количество заказов
)

In [5]:
df_sellers_pivot

purchase_month_number,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20
cohort_month,,,,,,,,,,,,,,,,,,,,
2017-01-01,176.0,141.0,142.0,117.0,119.0,114.0,109.0,111.0,108.0,98.0,102.0,101.0,96.0,96.0,97.0,89.0,78.0,76.0,54.0,24.0
2017-02-01,228.0,149.0,129.0,140.0,110.0,117.0,120.0,106.0,112.0,108.0,102.0,97.0,85.0,90.0,95.0,75.0,76.0,49.0,27.0,NaN
2017-03-01,184.0,107.0,104.0,92.0,90.0,85.0,86.0,79.0,85.0,79.0,73.0,68.0,65.0,64.0,60.0,57.0,38.0,18.0,NaN,NaN
2017-04-01,105.0,64.0,52.0,54.0,55.0,50.0,43.0,52.0,48.0,44.0,34.0,38.0,33.0,31.0,26.0,17.0,9.0,NaN,NaN,NaN
2017-05-01,127.0,62.0,61.0,68.0,63.0,62.0,56.0,58.0,54.0,56.0,55.0,49.0,44.0,35.0,25.0,12.0,NaN,NaN,NaN,NaN
2017-06-01,77.0,31.0,35.0,33.0,31.0,26.0,30.0,28.0,18.0,21.0,22.0,19.0,17.0,12.0,5.0,NaN,NaN,NaN,NaN,NaN
2017-07-01,107.0,65.0,63.0,60.0,66.0,68.0,67.0,61.0,59.0,55.0,55.0,55.0,34.0,19.0,NaN,NaN,NaN,NaN,NaN,NaN
2017-08-01,121.0,66.0,64.0,73.0,59.0,59.0,66.0,56.0,50.0,52.0,49.0,29.0,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2017-09-01,112.0,61.0,61.0,57.0,60.0,50.0,53.0,50.0,51.0,42.0,19.0,9.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
df_gmv_pivot = df.pivot_table(
    index='cohort_month',   # Строки - месяца когорты
    columns='purchase_month_number',  # Столбцы - месяцы покупок
    values='gmv_paid',  # Значения - количество заказов
)

In [7]:
df_gmv_pivot

purchase_month_number,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20
cohort_month,,,,,,,,,,,,,,,,,,,,
2017-01-01,82381.46,108402.27,174984.71,137618.62,189521.28,158679.53,155174.90,149336.91,138082.34,154081.29,196151.64,156929.81,149766.62,148845.35,166290.20,133035.51,105827.80,89005.88,28095.55,8874.63
2017-02-01,107552.63,105506.64,79844.99,124410.97,91575.21,95416.73,104666.70,92548.02,90015.98,121699.63,121104.68,121515.15,91008.61,92985.04,90554.16,82884.56,65638.75,24369.18,9320.92,NaN
2017-03-01,89220.26,51316.98,90600.42,76279.67,68463.91,83460.53,88950.20,93733.55,114970.55,93484.38,103523.45,78726.35,83256.30,71383.68,74435.45,65916.27,23885.50,5519.98,NaN,NaN
2017-04-01,28545.87,34854.17,32486.61,34480.76,35681.69,33598.28,40511.74,51867.36,42775.01,42487.24,34476.70,39006.43,31524.75,35873.02,21130.90,5940.64,1109.87,NaN,NaN,NaN
2017-05-01,62108.72,42563.95,56739.45,58311.16,48565.28,53341.34,52178.02,46391.49,41755.74,37108.20,39684.28,31444.62,23319.71,20441.00,8982.13,8992.92,NaN,NaN,NaN,NaN
2017-06-01,17809.39,24284.11,18888.71,11039.01,11096.24,9692.23,14709.08,9181.83,8425.63,9353.50,10049.30,9071.32,7401.03,4119.59,941.34,NaN,NaN,NaN,NaN,NaN
2017-07-01,33607.21,40618.53,45953.47,44083.57,69009.47,70119.99,72969.49,58939.96,78025.81,79003.46,92248.54,63001.97,15123.13,5528.34,NaN,NaN,NaN,NaN,NaN,NaN
2017-08-01,71262.36,98577.98,69137.56,67074.54,55501.41,51666.40,44262.64,44460.61,42667.44,32100.70,28715.27,7660.29,3242.75,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2017-09-01,30071.02,21023.83,32758.21,22820.04,43487.68,36473.49,30973.58,30803.31,27471.69,21633.42,5203.96,1589.46,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
df_arpps_pivot = df.pivot_table(
    index='cohort_month',   # Строки - месяца когорты
    columns='purchase_month_number',  # Столбцы - месяцы покупок
    values='arpps_paid',  # Значения - количество заказов
)

In [9]:
df_arpps_pivot

purchase_month_number,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20
cohort_month,,,,,,,,,,,,,,,,,,,,
2017-01-01,468.076477,768.810426,1232.286690,1176.227521,1592.615798,1391.925702,1423.622936,1345.377568,1278.540185,1572.258061,1923.055294,1553.760495,1560.068958,1550.472396,1714.331959,1494.781011,1356.766667,1171.130000,520.287963,369.77625
2017-02-01,471.722061,708.098255,618.953411,888.649786,832.501909,815.527607,872.222500,873.094528,803.714107,1126.848426,1187.300784,1252.733505,1070.689529,1033.167111,953.201684,1105.127467,863.667763,497.330204,345.219259,NaN
2017-03-01,484.892717,479.597944,871.157885,829.126848,760.710111,981.888588,1034.304651,1186.500633,1352.594706,1183.346582,1418.129452,1157.740441,1280.866154,1115.370000,1240.590833,1156.425789,628.565789,306.665556,NaN,NaN
2017-04-01,271.865429,544.596406,624.742500,638.532593,648.758000,671.965600,942.133488,997.449231,891.146042,965.619091,1014.020588,1026.485000,955.295455,1157.194194,812.726923,349.449412,123.318889,NaN,NaN,NaN
2017-05-01,489.045039,686.515323,930.154918,857.517059,770.877460,860.344194,931.750357,799.853276,773.254444,662.646429,721.532364,641.726939,529.993409,584.028571,359.285200,749.410000,NaN,NaN,NaN,NaN
2017-06-01,231.290779,783.358387,539.677429,334.515455,357.943226,372.778077,490.302667,327.922500,468.090556,445.404762,456.786364,477.437895,435.354706,343.299167,188.268000,NaN,NaN,NaN,NaN,NaN
2017-07-01,314.086075,624.900462,729.420159,734.726167,1045.598030,1031.176324,1089.096866,966.228852,1322.471356,1436.426545,1677.246182,1145.490364,444.797941,290.965263,NaN,NaN,NaN,NaN,NaN,NaN
2017-08-01,588.945124,1493.605758,1080.274375,918.829315,940.701864,875.701695,670.646061,793.939464,853.348800,617.321154,586.025918,264.147931,294.795455,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2017-09-01,268.491250,344.652951,537.019836,400.351579,724.794667,729.469800,584.407170,616.066200,538.660588,515.081429,273.892632,176.606667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
df_margin_pivot = df.pivot_table(
    index='cohort_month',   # Строки - месяца когорты
    columns='purchase_month_number',  # Столбцы - месяцы покупок
    values='margin_paid',  # Значения - количество заказов
)

In [11]:
df_margin_pivot

purchase_month_number,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20
cohort_month,,,,,,,,,,,,,,,,,,,,
2017-01-01,21244.7335,29019.1298,47042.2519,35870.6588,53845.3047,42922.5355,43431.6971,41139.3378,38808.7125,44878.6358,54953.3019,45171.8446,42225.9359,38473.8021,44086.8985,35454.6800,28599.5386,23825.3054,7430.6694,2481.4237
2017-02-01,25300.4309,26383.0883,19493.3747,30287.9108,22149.6630,23466.5935,25868.1703,22750.9695,22673.2475,30603.0821,30551.5282,30700.7950,22556.3070,22659.7062,22614.0447,21363.5772,17355.4067,6558.5516,2444.7130,NaN
2017-03-01,20500.4483,12733.7721,21629.3736,19282.3903,17435.4285,22538.7901,24751.1898,25418.4650,29946.5897,26579.4932,28183.6775,21810.5543,22903.8133,19823.0712,20570.0304,18883.4865,6155.2075,1403.0500,NaN,NaN
2017-04-01,7641.3728,9600.1913,8828.1514,9542.8226,10085.8977,9426.6075,11707.0019,14205.8217,11434.9392,11240.7381,9299.7783,10841.2170,8847.3178,10285.1687,5790.3036,1732.4410,316.8485,NaN,NaN,NaN
2017-05-01,15248.9067,9905.3712,13876.5383,14699.9521,12626.1605,12367.2626,14113.2719,12816.8940,10760.9185,9167.3793,9983.4830,8224.3396,6105.4545,5112.5400,2189.3620,1405.3510,NaN,NaN,NaN,NaN
2017-06-01,4602.3339,6323.8904,4651.4890,2865.0627,2865.3001,2377.7554,3436.8426,2170.8158,1808.8523,2309.3325,2579.4122,2242.2432,1835.9626,1065.6560,241.1807,NaN,NaN,NaN,NaN,NaN
2017-07-01,8526.8770,9646.6317,11469.9033,10587.8656,17214.2714,18128.6561,18991.5061,14446.6875,19700.1277,19660.8414,23549.6470,15691.3501,3804.7571,1420.4714,NaN,NaN,NaN,NaN,NaN,NaN
2017-08-01,14042.6998,16225.2577,12388.8438,12238.7837,10590.7101,10577.0840,10106.1880,10125.0109,9372.3514,7638.3941,6614.2314,1747.5809,569.8454,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2017-09-01,7937.2379,5347.2710,8700.8379,5919.2821,11257.1391,9365.5556,8167.0885,7930.1206,7205.5775,5632.2663,1291.9446,399.8182,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### Считаем сколько GMV приностит 1 селлер за год

первый способ, математически верный

In [12]:
#сначала считаем сумму GMV за год для когорт, которые живут с нами 12 месяцев
df_gmv_pivot.iloc[:9, :12].T.sum()

cohort_month
2017-01-01    1801344.76
2017-02-01    1255857.33
2017-03-01    1032730.25
2017-04-01     450771.86
2017-05-01     570192.25
2017-06-01     153600.35
2017-07-01     747581.47
2017-08-01     613087.20
2017-09-01     304309.69
dtype: float64

In [14]:
#затем считаем размер когорт
df_sellers_pivot.iloc[:9, :1].T.sum()

cohort_month
2017-01-01    176.0
2017-02-01    228.0
2017-03-01    184.0
2017-04-01    105.0
2017-05-01    127.0
2017-06-01     77.0
2017-07-01    107.0
2017-08-01    121.0
2017-09-01    112.0
dtype: float64

In [20]:
#делим сумму GMV за год для когорт, которые живут с нами 12 месяцев на размер когорты
df_gmv_pivot.iloc[:9, :12].T.sum()/df_sellers_pivot.iloc[:9, :1].T.sum()

cohort_month
2017-01-01    10234.913409
2017-02-01     5508.146184
2017-03-01     5612.664402
2017-04-01     4293.065333
2017-05-01     4489.702756
2017-06-01     1994.809740
2017-07-01     6986.742710
2017-08-01     5066.836364
2017-09-01     2717.050804
dtype: float64

втоорой способ, математически неверный

In [21]:
#суммируем для когорт, которые живут с нами 12 месяцев, месячные ARPPS
df_arpps_pivot.iloc[:9, :12].T.sum()

cohort_month
2017-01-01    15726.557153
2017-02-01    10451.366880
2017-03-01    11739.990559
2017-04-01     9237.313967
2017-05-01     9125.217801
2017-06-01     5285.508095
2017-07-01    12116.867381
2017-08-01     9683.487459
2017-09-01     5709.494767
dtype: float64

#### расчет LTV (здесь Value - это маржа) по когортам за 12 месяцев жизни

In [19]:
df_margin_pivot.iloc[:9, :12].T.sum()/df_sellers_pivot.iloc[:9, :1].T.sum()

cohort_month
2017-01-01    2831.409909
2017-02-01    1360.652868
2017-03-01    1471.794415
2017-04-01    1179.567043
2017-05-01    1132.208486
2017-06-01     496.536755
2017-07-01    1753.405279
2017-08-01    1005.513519
2017-09-01     706.733387
dtype: float64